# 📊 Supply Chain Data Exploration

This notebook provides a comprehensive exploration of the supply chain datasets used in this platform.

## Contents
1. Dataset Overview
2. Data Quality Analysis
3. Feature Distribution Analysis
4. Correlation Analysis
5. Time Series Analysis
6. Key Insights Summary

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

# Data paths
DATA_DIR = Path('../../data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
SAMPLES_DIR = DATA_DIR / 'samples'

print("Paths configured:")
print(f"  Raw data: {RAW_DIR}")
print(f"  Processed data: {PROCESSED_DIR}")
print(f"  Sample data: {SAMPLES_DIR}")

## 2. Load Datasets

In [ ]:
# Load available datasets
datasets = {}

# Load from samples directory
for csv_file in SAMPLES_DIR.glob('*.csv'):
    name = csv_file.stem
    datasets[name] = pd.read_csv(csv_file)
    print(f"Loaded {name}: {datasets[name].shape}")

# Load from processed directory if available
if PROCESSED_DIR.exists():
    for csv_file in PROCESSED_DIR.glob('*.csv'):
        name = f"processed_{csv_file.stem}"
        datasets[name] = pd.read_csv(csv_file)
        print(f"Loaded {name}: {datasets[name].shape}")

## 3. Dataset Overview

In [ ]:
def dataset_overview(df, name):
    """Generate comprehensive overview of a dataset."""
    print(f"\n{'='*60}")
    print(f"📊 Dataset: {name}")
    print(f"{'='*60}")
    print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"\nColumn Types:")
    print(df.dtypes.value_counts())
    print(f"\nMissing Values:")
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing.sum() > 0 else "  No missing values")
    print(f"\nFirst 5 Rows:")
    display(df.head())
    print(f"\nStatistical Summary:")
    display(df.describe())

# Show overview for each dataset
for name, df in datasets.items():
    dataset_overview(df, name)

## 4. Data Quality Analysis

In [ ]:
def analyze_data_quality(df, name):
    """Analyze data quality issues."""
    print(f"\n🔍 Data Quality Report: {name}")
    print("-" * 40)
    
    # Missing values
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    if missing_pct.max() > 0:
        print(f"\n⚠️ Columns with missing values:")
        print(missing_pct[missing_pct > 0])
    else:
        print("\n✅ No missing values")
    
    # Duplicates
    dup_count = df.duplicated().sum()
    print(f"\n📋 Duplicate rows: {dup_count} ({dup_count/len(df)*100:.2f}%)")
    
    # Outliers (for numeric columns)
    print(f"\n📊 Potential outliers (values > 3 std from mean):")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        z_scores = np.abs((df[col] - df[col].mean()) / df[col].std())
        outliers = (z_scores > 3).sum()
        if outliers > 0:
            print(f"  {col}: {outliers} outliers")

for name, df in datasets.items():
    analyze_data_quality(df, name)

## 5. Feature Distribution Analysis

In [ ]:
def plot_distributions(df, name, max_cols=6):
    """Plot distributions of numeric features."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns[:max_cols]
    
    if len(numeric_cols) == 0:
        print(f"No numeric columns in {name}")
        return
    
    n_cols = min(3, len(numeric_cols))
    n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    fig.suptitle(f'Feature Distributions: {name}', fontsize=14, fontweight='bold')
    
    axes = np.array(axes).flatten()
    
    for i, col in enumerate(numeric_cols):
        ax = axes[i]
        df[col].hist(ax=ax, bins=30, edgecolor='black', alpha=0.7)
        ax.set_title(col)
        ax.set_xlabel('')
    
    # Hide empty subplots
    for i in range(len(numeric_cols), len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()

for name, df in datasets.items():
    plot_distributions(df, name)

## 6. Correlation Analysis

In [ ]:
def plot_correlation_matrix(df, name):
    """Plot correlation matrix heatmap."""
    numeric_df = df.select_dtypes(include=[np.number])
    
    if numeric_df.shape[1] < 2:
        print(f"Not enough numeric columns for correlation in {name}")
        return
    
    corr = numeric_df.corr()
    
    # Create mask for upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool))
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, ax=ax, square=True, linewidths=0.5)
    ax.set_title(f'Correlation Matrix: {name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Show top correlations
    print(f"\n🔗 Top 10 Correlations (excluding self):")
    corr_pairs = corr.unstack().sort_values(ascending=False)
    corr_pairs = corr_pairs[corr_pairs < 1].drop_duplicates()
    print(corr_pairs.head(10))

for name, df in datasets.items():
    plot_correlation_matrix(df, name)

## 7. Key Insights Summary

In [ ]:
print("\n" + "="*60)
print("📋 KEY INSIGHTS SUMMARY")
print("="*60)

for name, df in datasets.items():
    print(f"\n📊 {name}:")
    print(f"   - {df.shape[0]:,} records, {df.shape[1]} features")
    print(f"   - {df.select_dtypes(include=[np.number]).columns.shape[0]} numeric features")
    print(f"   - {df.select_dtypes(include=['object']).columns.shape[0]} categorical features")
    
    missing = df.isnull().sum().sum()
    if missing > 0:
        print(f"   - ⚠️ {missing} missing values require attention")
    else:
        print(f"   - ✅ Complete data (no missing values)")

print("\n" + "="*60)